In [ ]:
# ==============================================================================
# MYZ 305E - Artificial Intelligence for Geomatics Engineering
# Term Project: Autonomous GeoAI Agent for Disaster Logistics & Routing
# Author: Umut Dilmen | Date: May 2026 | Location: Tuzla, Istanbul
# ==============================================================================

# --- STEP 0: ENVIRONMENT SETUP ---
print(">>> Installing required libraries...")
!pip install -q osmnx geopandas langchain-google-genai python-dotenv

import os
import random
import math
import pandas as pd
import geopandas as gpd
import osmnx as ox
import networkx as nx
import folium
from folium.plugins import AntPath, HeatMap
from IPython.display import display, HTML
from langchain_core.tools import tool
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv

# --- STEP 1: SECURITY & API CONFIGURATION ---
load_dotenv()
try:
    from google.colab import userdata
    api_key = userdata.get('GOOGLE_API_KEY')
except:
    api_key = os.getenv("GOOGLE_API_KEY")

if not api_key:
    api_key = "YOUR_API_KEY_HERE"

agent_llm = ChatGoogleGenerativeAI(model="gemini-2.5-pro", google_api_key=api_key, temperature=0)

# --- STEP 2: AUTOMATIC DATA DOWNLOAD & LOADING ---
def download_and_load_data():
    """Automatically downloads datasets from GitHub if they are missing."""
    # Replace these links with your ACTUAL raw GitHub file links
    files = {
        "hastaneler.csv": "https://raw.githubusercontent.com/lolzmrlolz/GeoAI-Disaster-Routing-MYZ305E-Term-Project/refs/heads/main/hastaneler.csv?token=GHSAT0AAAAAAD4GUE5RIGLYSJLKIU5NVSD42PY27EQ",
        "tuzla_toplanma.csv": "https://raw.githubusercontent.com/lolzmrlolz/GeoAI-Disaster-Routing-MYZ305E-Term-Project/refs/heads/main/tuzla_toplanma.csv?token=GHSAT0AAAAAAD4GUE5RMKV4NNRCTYFM7QTI2PY3AIQ"
    }

    for filename, url in files.items():
        if not os.path.exists(filename):
            print(f">>> Downloading {filename}...")
            # We use !wget to download directly into Colab
            os.system(f"wget -q {url} -O {filename}")

    if not os.path.exists("hastaneler.csv") or not os.path.exists("tuzla_toplanma.csv"):
        print("CRITICAL ERROR: Data files could not be acquired. Please upload them manually.")
        return None, None

    # Load and format data
    df_h = pd.read_csv("hastaneler.csv")
    df_h = df_h[df_h['Ilce Adi'].str.upper() == 'TUZLA'].copy()
    valid_cats = ['Şehir Hastanesi', 'Özel Hastane', 'Devlet Hastanesi', 'Üniversite Hastanesi']
    df_h = df_h[df_h['Alt Kategori'].isin(valid_cats)].copy()
    h_gdf = gpd.GeoDataFrame(df_h, geometry=gpd.points_from_xy(df_h['Longitude'], df_h['Latitude']), crs="EPSG:4326")

    a_gdf = pd.read_csv("tuzla_toplanma.csv")
    a_gdf = gpd.GeoDataFrame(a_gdf, geometry=gpd.points_from_xy(a_gdf['BOYLAM'], a_gdf['ENLEM']), crs="EPSG:4326")
    return h_gdf, a_gdf

# Initializing datasets
hospitals_gdf, assembly_gdf = download_and_load_data()

# --- STEP 3: NETWORK SIMULATION ---
if assembly_gdf is not None:
    print(">>> Constructing Autonomous Disaster Network...")
    G_base = ox.graph_from_place("Tuzla, Istanbul, Turkey", network_type="drive")
    G_base = ox.truncate.largest_component(G_base, strongly=True)

    DAMAGE_CONSTANT = 10**9
    for u, v, k, data in G_base.edges(data=True, keys=True):
        data['penalty'] = data['length']

    damaged_edges = random.sample(list(G_base.edges(keys=True)), int(len(G_base.edges) * 0.08))
    risk_heat = []
    blocked_geoms = []

    for u, v, k in damaged_edges:
        if G_base.has_edge(u, v):
            for key in G_base[u][v]:
                G_base[u][v][key]['penalty'] = DAMAGE_CONSTANT
                if 'geometry' in G_base[u][v][key]:
                    geom = G_base[u][v][key]['geometry']
                    blocked_geoms.append(geom)
                    risk_heat.append([geom.centroid.y, geom.centroid.x, 1])
else:
    print("Execution halted: Missing assembly point data.")

# --- STEP 4: GEOAI AGENT TOOL ---
@tool
def optimize_evacuation_routing(assembly_id: int) -> str:
    """Agent tool for spatial disaster logistics analysis."""
    start_pt = assembly_gdf.iloc[assembly_id]
    area_name = start_pt.get('AD', f"Zone {assembly_id}")
    start_node = ox.distance.nearest_nodes(G_base, X=start_pt.geometry.x, Y=start_pt.geometry.y)

    m = folium.Map(location=[start_pt.geometry.y, start_pt.geometry.x], zoom_start=13, tiles='cartodbdark_matter')
    fg_route = folium.FeatureGroup(name="Rescue Corridors", show=True)
    fg_marker = folium.FeatureGroup(name="Strategic Points", show=True)

    analysis_res = []
    for idx, hosp in hospitals_gdf.iterrows():
        target_node = ox.distance.nearest_nodes(G_base, X=hosp.geometry.x, Y=hosp.geometry.y)
        try:
            p_weight = nx.shortest_path_length(G_base, start_node, target_node, weight='penalty')
            if p_weight < DAMAGE_CONSTANT:
                d_norm = nx.shortest_path_length(G_base, start_node, target_node, weight='length') / 1000
                analysis_res.append({'name': hosp['Saglik Tesisi Adi'], 'd_afet': p_weight/1000, 'd_norm': d_norm, 'target': target_node, 'coords': (hosp.geometry.y, hosp.geometry.x)})
        except: continue

    df_top = pd.DataFrame(analysis_res).sort_values('d_afet').head(3)
    final_list = df_top.to_dict('records')

    if not final_list:
        return f"<div style='color:red;'><b>ALERT:</b> Area {area_name} is ISOLATED.</div>"

    dashboard = f"<div style='background: #000; color: #39FF14; padding: 15px; border: 2px solid #39FF14; border-radius: 10px; font-family: monospace;'><h3>🚨 COMMAND: {area_name}</h3><table style='width:100%;'>"
    for i, res in enumerate(final_list):
        color = "#39FF14" if i == 0 else "#FFA500"
        path = nx.shortest_path(G_base, start_node, res['target'], weight='penalty')
        path_coords = []
        for u, v in zip(path[:-1], path[1:]):
            edge = min(G_base.get_edge_data(u, v).values(), key=lambda x: x['penalty'])
            if 'geometry' in edge: path_coords.extend(list(zip(edge['geometry'].xy[1], edge['geometry'].xy[0])))
            else: path_coords.extend([(G_base.nodes[u]['y'], G_base.nodes[u]['x']), (G_base.nodes[v]['y'], G_base.nodes[v]['x'])])
        AntPath(path_coords, color=color, weight=5, delay=1500).add_to(fg_route)
        folium.Marker(res['coords'], popup=res['name'], icon=folium.Icon(color="red", icon="plus")).add_to(fg_marker)

        eta = math.ceil((res['d_afet'] / 20) * 60 + 3)
        base_eta = math.ceil((res['d_norm'] / 40) * 60)
        loss = ((eta - base_eta) / base_eta) * 100
        dashboard += f"<tr><td>{res['name']}</td><td align='center'>{eta} min</td><td align='right'>+{loss:.1f}%</td></tr>"

    brief = agent_llm.invoke(f"As an autonomous spatial agent, justify choosing {final_list[0]['name']} (ETA: {eta} min, Efficiency Loss: +{loss:.1f}%) for {area_name} over other options. Focus on disaster resilience and avoiding blocked infrastructure. Max 2 sentences.").content
    dashboard += f"</table><p style='font-size:0.8em; color:#ccc;'><b>AI Logic:</b> {brief}</p></div>"

    folium.Marker([start_pt.geometry.y, start_pt.geometry.x], icon=folium.Icon(color="green", icon="star")).add_to(fg_marker)
    HeatMap(risk_heat, radius=15, blur=10).add_to(m)
    fg_route.add_to(m); fg_marker.add_to(m)
    m.save("final_mission.html")
    return dashboard

# --- STEP 5: MISSION START ---
# Added safety check for 'NoneType' error
if assembly_gdf is not None and not assembly_gdf.empty:
    rand_idx = random.randint(0, len(assembly_gdf) - 1)
    agent_orchestrator = agent_llm.bind_tools([optimize_evacuation_routing])
    print(f">>> Mission Initiated for Zone ID: {rand_idx}")
    mission_call = f"Analyze evacuation for Assembly ID {rand_idx}."
    ai_resp = agent_orchestrator.invoke(mission_call)

    for tool_call in ai_resp.tool_calls:
        if tool_call["name"] == "optimize_evacuation_routing":
            ui_output = optimize_evacuation_routing.invoke(tool_call["args"])
            display(HTML(ui_output))
    display(HTML(filename="final_mission.html"))
else:
    print(">>> ERROR: Datasets not loaded properly. Please check your GitHub links or upload files manually.")

>>> Installing required libraries...
>>> Downloading hastaneler.csv...
>>> Downloading tuzla_toplanma.csv...


EmptyDataError: No columns to parse from file